In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

master_df = pd.read_csv("../data/processed/master_features.csv")

print(f"Loaded {len(master_df)} companies")
print(f"Columns: {len(master_df.columns)}")

Loaded 503 companies
Columns: 45


In [13]:
def get_comps_table(ticker1, ticker2):
    
    def get_company(ticker):
        row = master_df[master_df['symbol'] == ticker]
        if row.empty:
            return None
        return row.iloc[0]
    
    c1 = get_company(ticker1)
    c2 = get_company(ticker2)
    
    if c1 is None or c2 is None:
        print("One or both tickers not found in master data.")
        return None
    
    metrics = {
        'Company': [c1['name'], c2['name']],
        'Sector': [c1['sector'], c2['sector']],
        'Current Price': [f"${c1['current_price']:,.2f}", f"${c2['current_price']:,.2f}"],
        'Market Cap': [f"${c1['market_cap']/1e9:,.1f}B", f"${c2['market_cap']/1e9:,.1f}B"],
        'Enterprise Value': [f"${c1['enterprise_value']/1e9:,.1f}B", f"${c2['enterprise_value']/1e9:,.1f}B"],
        'Revenue': [f"${c1['revenue']/1e9:,.1f}B", f"${c2['revenue']/1e9:,.1f}B"],
        'Gross Margin': [f"{c1['gross_margin']*100:,.1f}%" if pd.notna(c1['gross_margin']) else 'N/A', f"{c2['gross_margin']*100:,.1f}%" if pd.notna(c2['gross_margin']) else 'N/A'],
        'EBITDA Margin': [f"{c1['ebitda_margin']*100:,.1f}%" if pd.notna(c1['ebitda_margin']) else 'N/A', f"{c2['ebitda_margin']*100:,.1f}%" if pd.notna(c2['ebitda_margin']) else 'N/A'],
        'Net Margin': [f"{c1['net_margin']*100:,.1f}%" if pd.notna(c1['net_margin']) else 'N/A', f"{c2['net_margin']*100:,.1f}%" if pd.notna(c2['ebitda_margin']) else 'N/A'],
        'P/E Ratio': [f"{c1['pe_ratio']:,.2f}" if pd.notna(c1['pe_ratio']) else 'N/A', f"{c2['pe_ratio']:,.2f}" if pd.notna(c2['pe_ratio']) else 'N/A'],
        'Forward P/E': [f"{c1['forward_pe']:,.2f}" if pd.notna(c1['forward_pe']) else 'N/A', f"{c2['forward_pe']:,.2f}" if pd.notna(c2['forward_pe']) else 'N/A'],
        'EV/EBITDA': [f"{c1['ev_ebitda']:,.2f}" if pd.notna(c1['ev_ebitda']) else 'N/A', f"{c2['ev_ebitda']:,.2f}" if pd.notna(c2['ev_ebitda']) else 'N/A'],
        'EV/Revenue': [f"{c1['ev_revenue']:,.2f}" if pd.notna(c1['ev_revenue']) else 'N/A', f"{c2['ev_revenue']:,.2f}" if pd.notna(c2['ev_revenue']) else 'N/A'],
        'P/S Ratio': [f"{c1['ps_ratio']:,.2f}" if pd.notna(c1['ps_ratio']) else 'N/A', f"{c2['ps_ratio']:,.2f}" if pd.notna(c2['ps_ratio']) else 'N/A'],
        'P/B Ratio': [f"{c1['pb_ratio']:,.2f}" if pd.notna(c1['pb_ratio']) else 'N/A', f"{c2['pb_ratio']:,.2f}" if pd.notna(c2['pb_ratio']) else 'N/A'],
        'P/FCF': [f"{c1['p_fcf']:,.2f}" if pd.notna(c1['p_fcf']) else 'N/A', f"{c2['p_fcf']:,.2f}" if pd.notna(c2['p_fcf']) else 'N/A'],
        'ROE': [f"{c1['roe']*100:,.1f}%" if pd.notna(c1['roe']) else 'N/A', f"{c2['roe']*100:,.1f}%" if pd.notna(c2['roe']) else 'N/A'],
        'ROA': [f"{c1['roa']*100:,.1f}%" if pd.notna(c1['roa']) else 'N/A', f"{c2['roa']*100:,.1f}%" if pd.notna(c2['roa']) else 'N/A'],
        'Debt/Equity': [f"{c1['debt_to_equity']:,.2f}" if pd.notna(c1['debt_to_equity']) else 'N/A', f"{c2['debt_to_equity']:,.2f}" if pd.notna(c2['debt_to_equity']) else 'N/A'],
        'Current Ratio': [f"{c1['current_ratio']:,.2f}" if pd.notna(c1['current_ratio']) else 'N/A', f"{c2['current_ratio']:,.2f}" if pd.notna(c2['current_ratio']) else 'N/A'],
        'Beta': [f"{c1['beta']:,.2f}" if pd.notna(c1['beta']) else 'N/A', f"{c2['beta']:,.2f}" if pd.notna(c2['beta']) else 'N/A'],
        '52W Position': [f"{c1['fifty_two_week_position']:,.1f}%" if pd.notna(c1['fifty_two_week_position']) else 'N/A', f"{c2['fifty_two_week_position']:,.1f}%" if pd.notna(c2['fifty_two_week_position']) else 'N/A'],
        'Dividend Yield': [f"{c1['dividend_yield']:,.2f}%" if pd.notna(c1['dividend_yield']) else 'N/A', f"{c2['dividend_yield']:,.2f}%" if pd.notna(c2['dividend_yield']) else 'N/A'],
        'Altman Z-Score': [f"{c1['altman_z_score']:,.3f}" if pd.notna(c1['altman_z_score']) else 'N/A', f"{c2['altman_z_score']:,.3f}" if pd.notna(c2['altman_z_score']) else 'N/A'],
        'Z-Score Zone': [c1['z_score_zone'] if pd.notna(c1['z_score_zone']) else 'N/A', c2['z_score_zone'] if pd.notna(c2['z_score_zone']) else 'N/A'],
        'Analyst Target': [f"${c1['analyst_target_price']:,.2f}" if pd.notna(c1['analyst_target_price']) else 'N/A', f"${c2['analyst_target_price']:,.2f}" if pd.notna(c2['analyst_target_price']) else 'N/A'],
        'Recommendation': [c1['recommendation'] if pd.notna(c1['recommendation']) else 'N/A', c2['recommendation'] if pd.notna(c2['recommendation']) else 'N/A'],
    }
    
    comps_df = pd.DataFrame(metrics, index=[ticker1, ticker2]).T
    
    return comps_df

In [14]:
comps = get_comps_table("AAPL", "MSFT")

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 100)

print(comps.to_string())

                        AAPL                   MSFT
Company           Apple Inc.  Microsoft Corporation
Sector            Technology             Technology
Current Price        $312.40                $418.71
Market Cap         $4,588.3B              $3,110.3B
Enterprise Value   $4,587.3B              $3,226.9B
Revenue              $451.4B                $318.3B
Gross Margin           47.9%                  68.3%
EBITDA Margin          35.4%                  58.0%
Net Margin             27.2%                  39.3%
P/E Ratio              37.87                  24.95
Forward P/E            32.51                  21.64
EV/EBITDA              28.68                  17.49
EV/Revenue             10.16                  10.14
P/S Ratio              10.16                   9.77
P/B Ratio              43.03                   7.51
P/FCF                  45.39                  84.04
ROE                   141.5%                  34.0%
ROA                    26.2%                  14.8%
Debt/Equity 

In [15]:
features_for_knn = [
    'pe_ratio', 'forward_pe', 'ps_ratio', 'pb_ratio', 'ev_ebitda',
    'ev_revenue', 'gross_margin', 'ebitda_margin', 'net_margin',
    'roe', 'roa', 'debt_to_equity', 'current_ratio', 'beta',
    'revenue', 'market_cap', 'fifty_two_week_position'
]

knn_df = master_df[['symbol', 'name', 'sector'] + features_for_knn].copy()
knn_df = knn_df.dropna(subset=features_for_knn)

scaler = StandardScaler()
knn_scaled = scaler.fit_transform(knn_df[features_for_knn])

knn_model = NearestNeighbors(n_neighbors=6, metric='euclidean')
knn_model.fit(knn_scaled)

print(f"KNN model trained on {len(knn_df)} companies")

KNN model trained on 464 companies


In [19]:
def find_similar_companies(ticker, n=5, same_sector=True):
    if ticker not in knn_df['symbol'].values:
        print(f"{ticker} not found in KNN dataset.")
        return None
    
    idx = knn_df[knn_df['symbol'] == ticker].index[0]
    ticker_sector = knn_df.loc[idx, 'sector']
    knn_idx = knn_df.index.get_loc(idx)
    
    if same_sector:
        sector_df = knn_df[knn_df['sector'] == ticker_sector].copy()
        sector_scaled = scaler.transform(sector_df[features_for_knn])
        
        sector_model = NearestNeighbors(n_neighbors=min(n+1, len(sector_df)), metric='euclidean')
        sector_model.fit(sector_scaled)
        
        ticker_sector_idx = sector_df.index.get_loc(idx)
        distances, indices = sector_model.kneighbors([sector_scaled[ticker_sector_idx]])
        
        similar = sector_df.iloc[indices[0][1:n+1]][['symbol', 'name', 'sector', 'market_cap', 'pe_ratio', 'ev_ebitda', 'gross_margin']].copy()
    else:
        distances, indices = knn_model.kneighbors([knn_scaled[knn_idx]])
        similar = knn_df.iloc[indices[0][1:n+1]][['symbol', 'name', 'sector', 'market_cap', 'pe_ratio', 'ev_ebitda', 'gross_margin']].copy()
    
    similar['distance'] = distances[0][1:n+1].round(3)
    similar['market_cap'] = similar['market_cap'].apply(lambda x: f"${x/1e9:,.1f}B")
    similar['pe_ratio'] = similar['pe_ratio'].apply(lambda x: f"{x:,.2f}")
    similar['ev_ebitda'] = similar['ev_ebitda'].apply(lambda x: f"{x:,.2f}")
    similar['gross_margin'] = similar['gross_margin'].apply(lambda x: f"{x*100:,.1f}%")
    
    return similar

In [20]:
test2 = find_similar_companies("JPM")
if test2 is not None:
    print("Companies most similar to JPMorgan:\n")
    print(test2.to_string())
else:
    print("JPM not available in KNN dataset — financial sector excluded.")

print()

test3 = find_similar_companies("XOM")
if test3 is not None:
    print("Companies most similar to ExxonMobil:\n")
    print(test3.to_string())

JPM not found in KNN dataset.
JPM not available in KNN dataset — financial sector excluded.

Companies most similar to ExxonMobil:

    symbol                            name  sector market_cap pe_ratio ev_ebitda gross_margin  distance
101    CVX             Chevron Corporation  Energy    $375.2B    32.82     11.03        42.4%     2.065
298    MPC  Marathon Petroleum Corporation  Energy     $77.6B    17.49     11.39        10.7%     3.142
367    PSX                     Phillips 66  Energy     $74.0B    18.22     13.77        12.5%     3.227
464    VLO       Valero Energy Corporation  Energy     $76.9B    18.91      9.31        14.6%     3.338
123    COP                  ConocoPhillips  Energy    $143.8B    20.00      6.95        45.6%     3.818


In [21]:
master_df.to_csv("../data/processed/master_features.csv", index=False)
knn_df.to_csv("../data/processed/knn_features.csv", index=False)

import pickle
with open("../data/processed/knn_model.pkl", "wb") as f:
    pickle.dump(knn_model, f)
with open("../data/processed/knn_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Saved:")
print("  master_features.csv")
print("  knn_features.csv")
print("  knn_model.pkl")
print("  knn_scaler.pkl")

Saved:
  master_features.csv
  knn_features.csv
  knn_model.pkl
  knn_scaler.pkl
